In [1]:
import sys
sys.path.append('../../..') #This line makes it possible to access scripts and settings, which are above notebooks in heirarchy
#the above line also makes it possible to access files from the root with out a ROOT_DIR
import subprocess
import importlib
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from numpy.lib.recfunctions import drop_fields, structured_to_unstructured

from notebooks.preprocessing.phase2preprocess import preprocess_general, load_parquet
# from scripts.generate_dataset import generate_dataset
from scripts.dataImport import impsettings

In [2]:
NORMAL_LABEL = "Benign"

In [3]:
#could we add these constants to a settings .toml file and settings class?
#We could also put all our model kwargs in settings classes and .toml files
BASE_DIR = Path("__file__").resolve().parent
ROOT_DIR = BASE_DIR.parents[1]
ALGORITHM_DIR = ROOT_DIR / "model" / "algorithms"
OUTPUT_DIR = ROOT_DIR / "saved_models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
MODEL_FILES = {
    "isolation_forest": "model_isolation_forest",
    "autoencoder": "model_autoencoder",
    "kmeans": "model_kmeans",
}

In [5]:
def import_module_or_notebook(name):
    py = ALGORITHM_DIR / f"{name}.py"
    nb = ALGORITHM_DIR / f"{name}.ipynb"

    if not py.exists() and not nb.exists():
        raise FileNotFoundError(f"{name}.py or {name}.ipynb not found")

    if nb.exists():
        subprocess.run(
            [sys.executable, "-m", "nbconvert", "--to", "script", "--output", name, str(nb)],
            cwd=ALGORITHM_DIR,
            check=True,
            capture_output=True,
            text=True,
        )

    if str(ALGORITHM_DIR) not in sys.path:
        sys.path.insert(0, str(ALGORITHM_DIR))

    importlib.invalidate_caches()
    return importlib.import_module(name) if name not in sys.modules else importlib.reload(sys.modules[name])

In [6]:
def get_models(names=None):
    if names is None:
        names = list(MODEL_FILES)
    if isinstance(names, str):
        names = [names]
    return {name: import_module_or_notebook(MODEL_FILES[name]) for name in names}


In [7]:
# raw = load_parquet(impsettings.PROCESSED_DATA_PATH, 'combined_sample.parquet')

In [8]:
# exploredf = pd.DataFrame(raw)

In [9]:
# pkt_features = [feat for feat in raw.dtype.names if feat.startswith('pkt_')]

In [10]:
# exploredf[pkt_features]

In [11]:
def get_training_data():
    #create the dataset if it doesn't already exist
    input_data = impsettings.PROCESSED_DATA_PATH / "flow_training_sample.parquet"
    if not input_data.is_file():
        print('Constructing Datasets...')
        %run ../../../scripts/generate_dataset.py
    #import wwt from data/processed
    raw = load_parquet(impsettings.PROCESSED_DATA_PATH, 'combined_sample.parquet')
    #seperate out labels
    y = raw['pkt_label']
    raw = drop_fields(raw, 'pkt_label', asrecarray=True)
    #Seperate out temp and test data
    X_temp, X_test, y_temp, y_test = train_test_split(raw, y, test_size=0.15, stratify=y, random_state=42)

    #preprocess training data first
    print('preprocess training/validation data')
    X_temp_pp = preprocess_general(array=X_temp, split='Train')

    #seperate out packet features from wwt for training/testing
    pkt_features = [feat for feat in X_temp_pp.dtype.names if feat.startswith('pkt_')]
    X_temp_pp_flow = X_temp_pp[pkt_features]

    #preprocess testing data, using mean, std, and onehot labels learned from training
    print('preprocess testing data')
    X_test_pp = preprocess_general(array=X_test, split='Test')
    X_train, X_val, y_train, y_val = train_test_split(X_temp_pp_flow, y_temp, test_size=0.15, stratify=y_temp, random_state=42)

    #change to dataframes for compatability with model train functions.
    return (
        pd.DataFrame(X_train), 
        pd.DataFrame(X_val), 
        pd.DataFrame(X_test_pp), 
        pd.Series(y_train), 
        pd.Series(y_val), 
        pd.Series(y_test)
        )

In [12]:
def to_df(X):
    if isinstance(X, pd.DataFrame):
        return X.copy()
    X = np.asarray(X)
    return pd.DataFrame(X, columns=[f"feature_{i}" for i in range(X.shape[1])])

In [13]:
def train_model(name):
    X_train, X_val, _, y_train, y_val, _ = get_training_data()
    model_mod = get_models(name)[name]

    if name == "isolation_forest":
        return model_mod.train(
            X_train, y_train, X_val, y_val,
            normal_label=NORMAL_LABEL,
            model_kwargs={
                "n_estimators": 200,
                "max_samples": 256,
                "contamination": "auto",
                "max_features": 1.0,
                "random_state": 42,
            },
            tune_threshold=True,
            save_path=str(OUTPUT_DIR / "isolation_forest.joblib"),
        )

    if name == "autoencoder":
        return model_mod.train(
            X_train, y_train, X_val, y_val,
            normal_label=NORMAL_LABEL,
            model_kwargs={"hidden_dims": (64, 32, 16), "latent_dim": 8, "random_state": 42},
            train_kwargs={"num_epochs": 15, "batch_size": 512, "lr": 0.003},
            tune_threshold=True,
            save_path=str(OUTPUT_DIR / "autoencoder.pt"),
        )

    if name == "kmeans":
        return model_mod.train(
            X_train, y_train, X_val, y_val,
            normal_label=NORMAL_LABEL,
            model_kwargs={"n_clusters": 2, "random_state": 42, "n_init": "auto"},
            save_path=str(OUTPUT_DIR / "kmeans.joblib"),
        )

    raise ValueError(f"Unknown model: {name}")

In [14]:
def train_all():
    return {name: train_model(name) for name in MODEL_FILES}

In [15]:
def load_saved_model(name):
    if name == "isolation_forest":
        return joblib.load(OUTPUT_DIR / "isolation_forest.joblib")

    if name == "kmeans":
        return joblib.load(OUTPUT_DIR / "kmeans.joblib")

    if name == "autoencoder":
        ckpt = torch.load(OUTPUT_DIR / "autoencoder.pt", map_location="cpu", weights_only=False)
        auto_mod = get_models("autoencoder")["autoencoder"]
        model = auto_mod.build_autoencoder(ckpt["input_dim"], **ckpt["model_kwargs"])
        model.load_state_dict(ckpt["state_dict"])
        model.eval()
        return {
            "model": model,
            "threshold": ckpt["threshold"],
            "normal_label": ckpt["normal_label"],
        }

    raise ValueError(f"Unknown model: {name}")

In [16]:
def predict_one(X, name):
    X = to_df(X)
    X_np = X.to_numpy()
    obj = load_saved_model(name)

    if name == "isolation_forest":
        score = obj["model"].decision_function(X_np)
        pred_bin = (score < obj["threshold"]).astype(int)

    elif name == "autoencoder":
        Xt = torch.tensor(X_np, dtype=torch.float32)
        with torch.no_grad():
            out = obj["model"](Xt)
            recon = out[0] if isinstance(out, (tuple, list)) else out
            score = ((recon - Xt) ** 2).mean(dim=1).cpu().numpy()
        pred_bin = (score > obj["threshold"]).astype(int)

    elif name == "kmeans":
        dist = obj["model"].transform(X_np)
        score = dist.min(axis=1)
        threshold = obj.get("threshold", np.percentile(score, 95))
        pred_bin = (score > threshold).astype(int)

    else:
        raise ValueError(f"Unknown model: {name}")

    pred_lbl = np.where(pred_bin == 0, NORMAL_LABEL, "Attack")

    return {
        "label": pred_lbl,
        "binary": pred_bin,
        "score": score,
    }

In [17]:
def predict_all(X):
    X = to_df(X)

    score_cols = []
    label_cols = []
    votes = []

    for name in MODEL_FILES:
        result = predict_one(X, name)
        votes.append(result["binary"])
        score_cols.append(np.asarray(result["score"]))
        label_cols.append(np.asarray(result["label"], dtype=object))

    votes = np.vstack(votes).T
    final_bin = (votes.sum(axis=1) >= 2).astype(int)
    final_label = np.where(final_bin == 0, NORMAL_LABEL, "Attack")
    final_score = votes.mean(axis=1)

    X_array = X.to_numpy()
    # scores_array = np.column_stack(score_cols + [final_score]) for if you want to show scores for each model
    scores_array = np.column_stack(final_score)
    #labels_array = np.column_stack(label_cols + [final_label]) for if you want to show labels for each model
    labels_array = np.column_stack(final_label)

    return X_array, scores_array, labels_array

In [18]:
def predict_phase2(X, model_name=None, as_numpy=True):
    X = to_df(X)

    if model_name:
        result = predict_one(X, model_name)
        X_array = X.to_numpy()
        scores_array = np.asarray(result["score"]).reshape(-1, 1)
        labels_array = np.asarray(result["label"], dtype=object).reshape(-1, 1)
        return X_array, scores_array, labels_array

    return predict_all(X)

In [19]:
trained_models = train_all()

preprocess training/validation data


/Users/mackenziemallett/miniforge3/envs/CapstoneII/lib/python3.13/site-packages/pandas/core/nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


preprocess testing data
ISOLATION FOREST
Training on 144,499 benign samples (excluded 3,032 attack samples).
Isolation Forest fitted.
Threshold tuned: 0.03019  (val F1=0.0497)
              precision    recall  f1-score   support

      Benign       0.98      0.98      0.98     25500
      Attack       0.06      0.04      0.05       535

    accuracy                           0.96     26035
   macro avg       0.52      0.51      0.52     26035
weighted avg       0.96      0.96      0.96     26035

Isolation forest done training...
Saved > /Users/mackenziemallett/Desktop/Google Drive/MADS/Professional Foundations+Professional Career Development/Capstone ll/ECE597-Capstone-IoT-IDS/notebooks/saved_models/isolation_forest.joblib
preprocess training/validation data


/Users/mackenziemallett/miniforge3/envs/CapstoneII/lib/python3.13/site-packages/pandas/core/nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


preprocess testing data
AUTOENCODER
Epoch [1/15], Reconstruction Loss: 0.049935
Epoch [3/15], Reconstruction Loss: 0.017709
Epoch [6/15], Reconstruction Loss: 0.012066
Epoch [9/15], Reconstruction Loss: 0.010050
Epoch [12/15], Reconstruction Loss: 0.008687
Epoch [15/15], Reconstruction Loss: 0.008201
Threshold tuned: 0.079005  (val F1=0.6084)
              precision    recall  f1-score   support

      Benign       0.99      1.00      0.99     25500
      Attack       0.77      0.50      0.61       535

    accuracy                           0.99     26035
   macro avg       0.88      0.75      0.80     26035
weighted avg       0.99      0.99      0.99     26035

Autoencoder done training...
Saved > /Users/mackenziemallett/Desktop/Google Drive/MADS/Professional Foundations+Professional Career Development/Capstone ll/ECE597-Capstone-IoT-IDS/notebooks/saved_models/autoencoder.pt
preprocess training/validation data


/Users/mackenziemallett/miniforge3/envs/CapstoneII/lib/python3.13/site-packages/pandas/core/nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


preprocess testing data
KMEANS
Inertia: 11920554.0000 | Silhouette: 0.4108 | CH: 14794.6441
KMeans done training
Saved > /Users/mackenziemallett/Desktop/Google Drive/MADS/Professional Foundations+Professional Career Development/Capstone ll/ECE597-Capstone-IoT-IDS/notebooks/saved_models/kmeans.joblib


In [21]:
X_train, X_val, X_test, y_train, y_val, _ = get_training_data()
pkt_features = [feat for feat in X_test.columns if feat.startswith('pkt_')]
X_arr, score_arr, label_arr = predict_phase2(X_test[pkt_features])

preprocess training/validation data


/Users/mackenziemallett/miniforge3/envs/CapstoneII/lib/python3.13/site-packages/pandas/core/nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


preprocess testing data
